In [1]:
import re
import pandas
import joblib
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn import preprocessing

import nltk
from nltk.corpus import stopwords

In [2]:
# loading tainig dataset
df_train = pandas.read_csv("../dataset/training_clean.csv")

nltk.download("stopwords", quiet=True)
stopWords = stopwords.words("english")

df_train = df_train.drop(stopWords, axis=1, errors="ignore")
df_train.drop("Prediction", axis=1)
columns = df_train.columns

# loading testing dataset
df_test = pandas.read_csv("../dataset/testing_emails.csv")
df_test["email"] = df_test["email"].astype(str)

print(f"Shape of testing dataset: {df_test.shape}")

Shape of testing dataset: (3000, 2)


In [3]:
# data preprocessing

def containsEnglishOnly(text):
    return bool(re.match(r"^[a-zA-Z\s]*$", text))

# removing non-english words containig rows
df_test = df_test[df_test["email"].apply(containsEnglishOnly)]

# resetting index of dataframe
df_test = df_test.reset_index(drop=True)

print(f"Shape of testing dataset after preprocessing: {df_test.shape}")

Shape of testing dataset after preprocessing: (1916, 2)


In [4]:
# futher preprocessing and tokenization

def preprocess(text):
    text = re.sub(r'[^a-z\s]', '', text.lower())
    text = re.sub(r"\W", " ", text)
    text = re.sub(r'\s+', " ", text)

    tokens = text.split()
    return tokens

token_frequency = []

for email in df_test["email"]:
    tokens = preprocess(email)
    word_count = Counter(tokens)

     # Create a dictionary for the current email's word frequencies
    data = {word: word_count.get(word, 0) for word in columns}
    token_frequency.append(data)

In [5]:
# creating dataframe from processes text

x_axis = pandas.DataFrame(token_frequency)
y_axis = df_test.iloc[:,-1]
min_max_scaler = preprocessing.MinMaxScaler()

# transform the features in X_test_external using the fitted MinMaxScaler
x_axis = min_max_scaler.fit_transform(x_axis)

print(f"Shape of testing dataset: x{x_axis.shape}, y:{y_axis.shape}")

Shape of testing dataset: x(1916, 2868), y:(1916,)


In [10]:
# loading the Improved LR Model
Imp_LR_Model = joblib.load("../models/Imp_LR_Model.pkl")

# Test the LR_Model on x_axis
test = Imp_LR_Model.predict(x_axis[:, :-2])

print("Classification Report:")
print(classification_report(test, y_axis))


Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.78      0.81      1646
           1       0.11      0.16      0.13       270

    accuracy                           0.69      1916
   macro avg       0.48      0.47      0.47      1916
weighted avg       0.74      0.69      0.72      1916

